In [ ]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
# from model.systemModel                  import SystemModel as systemModel
from model.componentModel               import SystemModel as componentModel
from sklearn.preprocessing              import MinMaxScaler

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json
import package
import sys
import math


package.include()
outputDir = os.path.expanduser("~/Desktop/Dropbox/codeData/2ndRealDataJournalWithoutGit/")
# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

##### HI COMPONENT AND SYSTEM

In [ ]:
preData                 = PreProcessData(config = myConfig)                    # Preprocessing data

package.include()
outputDir       = os.path.expanduser("~/Desktop/Dropbox/codeData/2ndRealDataJournalWithoutGit/")
cr1, cr2, cv2   = preData.loadH5File(dir=outputDir + "data/", name = "AdequateConsistentData", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file

# -------------------------------
# Handle to make the length of 
# each component consistently
cr1 = preData.nanFilling(dfFiltered=cr1, validColumn=preData.crusher1Value)
cr2 = preData.nanFilling(dfFiltered=cr2, validColumn=preData.crusher2Value)
cv2 = preData.nanFilling(dfFiltered=cv2, validColumn=preData.conveyorValue + preData.crusher2ConveyorValue)
# insertedCr1, insertedCr2,  insertedCv2   = self.fillForTheSameLength(data         = [withHourCr1,withHourCr2,withHourCv2], 
#                                                                      validColumn  = [self.crusher1Value, self.crusher2Value, self.conveyorValue + self.crusher2ConveyorValue])


if preData.config["splitTrainTest"]:
    target_time = pd.Timestamp('2023-10-01 00:00:00')

    rowCr1 = cr1.index[pd.to_datetime(cr1['tiempo']) == target_time][0]

    rowCr2 = cr2.index[pd.to_datetime(cr2['tiempo']) == target_time][0]

    rowCv2 = cv2.index[pd.to_datetime(cv2['tiempo']) == target_time][0]

    # Display the row
    print(rowCr1, rowCr2, rowCv2)

    trainCr1 = cr1.iloc[:rowCr1]
    trainCr2 = cr2.iloc[:rowCr2]
    trainCv2 = cv2.iloc[:rowCv2]
else:
    trainCr1 = cr1
    trainCr2 = cr2
    trainCv2 = cv2

# --------------------------------
# Add hour column into the file
# print(cv2.columns)
withHourCr1             = preData.addHours(trainCr1.copy())
withHourCr2             = preData.addHours(trainCr2.copy())
withHourCv2             = preData.addHours(trainCv2.copy())

# if self.config["component"]  == "system":

# else:
#     insertedCr1 = withHourCr1
#     insertedCr2 = withHourCr2
#     insertedCv2 = withHourCv2
# --------------------------------
# Sumarry data (option)
# _,mofCr1                = self.sumarryDataFromFile(withHourCr1, printUsefulMOF = True, printMOFTable = True,typeOfTime= "hours")
# _                       = self.sumarryDataFromFile(withHourCr2, printUsefulMOF = True, printMOFTable = True,typeOfTime= "hours")
# _                       = self.sumarryDataFromFile(withHourCv2, printUsefulMOF = True, printMOFTable = True,typeOfTime= "hours")

withIDResults, mofOrder, shutdownMask = preData.attachIDPerEvent(df = [withHourCr1,withHourCr2,withHourCv2])
# withIDCr1, withIDCr2, withIDCv2, IdSys

print("--------------------WITHOUT MAINTENANCE--------------------------")
print(withIDResults["cr1"]['trajIDWithoutM'])
print(withIDResults["cr2"]['trajIDWithoutM'])
print(withIDResults["cv2"]['trajIDWithoutM'])

print("------------------WITH MAINTENANCE----------------------------")
print(withIDResults["cr1"]['trajIDWithM'])
print(withIDResults["cr2"]['trajIDWithM'])
print(withIDResults["cv2"]['trajIDWithM'])
# print(withIDResults["sys"])

addedData               = preData.addition_var(withIDResults["cr1"]) 
handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1

# -----------------------------
# Normalize
normalizedCr1           = preData.normalizeBuilinFunction(addedData,            headline = handledValues)  
normalizedCr2           = preData.normalizeBuilinFunction(withIDResults["cr2"], headline = preData.crusher2Value)  
normalizedCv2           = preData.normalizeBuilinFunction(withIDResults["cv2"], headline = preData.conveyorValue + preData.crusher2Value)  

# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCr1         = preData.addCycleAndRul(normalizedCr1, shutdownMask["cr1"])                        # Add the time to the data
withTimeDataCr2         = preData.addCycleAndRul(normalizedCr2, shutdownMask["cr2"])                        # Add the time to the data
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2, shutdownMask["cv2"])                        # Add the time to the data

# aiLogger.info("Export the file to test!!!")
# withTimeDataCr1.to_excel("IDCrusher1.xlsx", index=False)

# -----------------------------
# Add the cycle and RUL of system
systemData                      = pd.DataFrame()
systemData['hours']             = withTimeDataCr1["hours"]
systemData['trajIDWithoutM']    = withIDResults["sysIDWithoutM"]
systemData['trajIDWithM']       = withIDResults["sysIDWithM"]
withTimeDataSys                 = preData.addCycleAndRul(systemData)

# -----------------------------
# Extract before splitting
timeLabel               = ["hours", "tFailureWithMaintenance", "tFailureWithoutMaintenance", "tBreak"]
dataCr1                 = withTimeDataCr1[timeLabel + handledValues]
cycleCr1                = withTimeDataCr1["tFailureWithoutMaintenance"]
rulCr1                  = withTimeDataCr1["rul"]
trajectIDCr1            = withTimeDataCr1["trajIDWithM"]
timeCr1                 = withTimeDataCr1["hours"]

dataCr2                 = withTimeDataCr2[timeLabel + preData.crusher2Value]
cycleCr2                = withTimeDataCr2["tFailureWithoutMaintenance"]
rulCr2                  = withTimeDataCr2["rul"]
trajectIDCr2            = withTimeDataCr2["trajIDWithM"]

dataCv2                 = withTimeDataCv2[timeLabel + preData.conveyorValue + preData.crusher2ConveyorValue]
cycleCv2                = withTimeDataCv2["tFailureWithoutMaintenance"]
rulCv2                  = withTimeDataCv2["rul"]
trajectIDCv2            = withTimeDataCv2["trajIDWithM"]

systemRUL               = withTimeDataSys["rul"]


# print(timeCr1)
# print(timeCr2)
# print(timeCv2)
# print(insertedCr1Sys["hours"])



### Crusher1

In [3]:
def calculate_mae_for_numpy(list1, list2):
    if len(list1) != len(list2):
        raise ValueError("The two lists must have the same length.")
    
    maes = []  # To store the MAE for each pair of arrays
    for arr1, arr2 in zip(list1, list2):
        if arr1.shape != arr2.shape:
            raise ValueError("Each pair of arrays must have the same shape.")
        mae = np.mean(np.abs(arr1 - arr2))  # Element-wise absolute difference, then mean
        maes.append(mae)
    
    overall_mae = np.mean(maes)  # Average MAE across all pairs
    return overall_mae

In [ ]:
myConfig["component"] = "Cr1"

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCr1"]))  # Batch size is None, feature size is 32()
model.summary()
# model.load_weights(model_dir = "checkpoint/bestModel37/")
model.load_weights(model_dir = outputDir + "checkpoint/crusher1Best_Trendability_SensorSelection/")

In [ ]:
results             = {}
time                = {}
results["cr1"]      = []
results["rulCr1"]   = []
time["timeCr1"]     = []
listRUL             = []

probsAll        = []
binaryMaskAll   = []
trajectID       = trajectIDCr1.astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr1, seqTimeCr1,seqRUL           = preData.getTensorData(dataList   =    [dataCr1, timeCr1,rulCr1], 
                                                                    trajectId  = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr1)
    # print(predictedRul)
    results["cr1"].append(pd.Series(tf.squeeze(zCr1).numpy()))
    results["rulCr1"].append(predictedRul)
    time["timeCr1"].append(seqTimeCr1[:,-1,:])
    listRUL.append(seqRUL[:,-1,:])


print(listRUL[0].shape, results["rulCr1"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rulCr1"])
print("Crusher 1 MAE:", mae)

print(np.max(np.concatenate(listRUL)), np.min(np.concatenate(listRUL)))

# temp_df = pd.concat([rulCr1, rulCr2, rulCv2], axis=1)
# # print(np.max(np.concatenate(rulCr1, rulCr2 , rulCv2)), np.min(np.concatenate(rulCr1 , rulCr2, rulCv2)))
# print(temp_df.max().max(),temp_df.min().min())

In [ ]:
print(time["timeCr1"][0].shape)

### Crusher2

In [ ]:
myConfig["component"] = "Cr2"

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCr2"]))  # Batch size is None, feature size is 32()
model.summary()
# model.load_weights(model_dir = "checkpoint/bestModel38/")
model.load_weights(model_dir = outputDir + "checkpoint/crusher2Best_Trendability_SensorSelection/")

In [ ]:
results["cr2"]      = []
results["rulCr2"]   = []
time["timeCr2"]     = []
listRUL             = []

probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCr2["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr2, seqTimeCr2,seqRUL          = preData.getTensorData(dataList     =    [dataCr2, timeCr2, rulCr2], 
                                                                    trajectId   = trajectID,
                                                                    seqLength   = preData.config['seqLength'], 
                                                                    overlap     = preData.config['seqLength']-1,
                                                                    getAtIdx    = idx + 1,
                                                                    typeInput   = "listOfPandas")
    
    zCr2, binaryMaskCr2, probsMaskCr2, predictedRul     = model.getHIandRUL(seqDataCr2)
    # print(predictedRul)
    results["cr2"].append(pd.Series(tf.squeeze(zCr2).numpy()))
    results["rulCr2"].append(predictedRul)
    time["timeCr2"].append(seqTimeCr2[:,-1,:])
    listRUL.append(seqRUL[:,-1,:])


print(listRUL[0].shape, results["rulCr2"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rulCr2"])
print("Crusher 2 MAE:", mae)

print(np.max(np.concatenate(listRUL)), np.min(np.concatenate(listRUL)))


###### Conveyorbelt 2

In [ ]:
myConfig["component"] = "Cv2"

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCv2"]))  # Batch size is None, feature size is 32()
model.summary()
# model.load_weights(model_dir = "checkpoint/bestModel41/")
model.load_weights(model_dir = outputDir +  "checkpoint/CV2Best_Trendability_SensorSelection/")

In [ ]:
results["cv2"]          = []
results["rulCv2"]       = []
time["timeCv2"]         = []
listRUL                 = []

probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCv2["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCv2, seqTimeCv2,seqRUL            = preData.getTensorData(dataList   =    [dataCv2, timeCv2, rulCv2], 
                                                                    trajectId  = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCv2, binaryMaskCv2, probsMaskCv2, predictedRul     = model.getHIandRUL(seqDataCv2)
    # print(predictedRul)
    results["cv2"].append(pd.Series(tf.squeeze(zCv2).numpy()))
    results["rulCv2"].append(predictedRul)
    time["timeCv2"].append(seqTimeCv2[:,-1,:])
    listRUL.append(seqRUL[:,-1,:])


print(listRUL[0].shape, results["rulCv2"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rulCv2"])
print("Conveyor belt 2 MAE:", mae)

print(np.max(np.concatenate(listRUL)), np.min(np.concatenate(listRUL)))


###### System

In [11]:
# # ------------------------------
# # Normalize
# # ------------------------------
# combined_data = pd.concat(results["cr1"], ignore_index=True)
# data_reshaped = combined_data.values.reshape(-1, 1)
# scalerCr1       = MinMaxScaler()
# scalerCr1.fit(data_reshaped)
# results["cr1"]  = [pd.DataFrame(scalerCr1.transform(df.values.reshape(-1, 1))) for df in results["cr1"]]


# combined_data = pd.concat(results["cr2"], ignore_index=True)
# data_reshaped = combined_data.values.reshape(-1, 1)
# scalerCr2       = MinMaxScaler()
# scalerCr2.fit(data_reshaped)
# results["cr2"]  = [pd.DataFrame(scalerCr2.transform(df.values.reshape(-1, 1))) for df in results["cr2"]]

# combined_data = pd.concat(results["cv2"], ignore_index=True)
# data_reshaped = combined_data.values.reshape(-1, 1)
# scalerCv2       = MinMaxScaler()
# scalerCv2.fit(data_reshaped)
# results["cv2"]  = [pd.DataFrame(scalerCv2.transform(df.values.reshape(-1, 1))) for df in results["cv2"]]

In [12]:
# scalerCr1       = MinMaxScaler()
# norHI1          = scalerCr1.fit_transform(aligned_HIs[1])

# scalerCr2       = MinMaxScaler()
# norHI2          = scalerCr1.fit_transform(aligned_HIs[2])

# scalerCv2       = MinMaxScaler()
# norHI3          = scalerCv2.fit_transform(aligned_HIs[3])

In [13]:
# ------------------------------
# HI system
all_times = insertedCr1Sys["hours"]

# Create a dictionary to store aligned HIs for each component
aligned_HIs = {1: [], 2: [], 3: []}


def align_hi(HI, timeHI, all_times):
    aligned = pd.Series(index=all_times, dtype=float)       # Initialize with NaN
    for hi, times in zip(HI, timeHI):
        aligned.loc[times[:,0]] = hi.values                 # Align based on the original time
    return aligned

# Align each component's HI

aligned_HIs[1] = align_hi(results["cr1"], time["timeCr1"], all_times)
aligned_HIs[2] = align_hi(results["cr2"], time["timeCr2"], all_times)
aligned_HIs[3] = align_hi(results["cv2"], time["timeCv2"], all_times)


# Combine all aligned HIs into a DataFrame
aligned_df = pd.DataFrame({'Time'           : all_times,
                           'OriginalHICr1'  : aligned_HIs[1].values,
                           'OriginalHICr2'  : aligned_HIs[2].values,
                           'OriginalHICv2'  : aligned_HIs[3].values})


aligned_df["trajectoryCr1"] = withIDCr1Sys["trajectoryID"]
aligned_df["trajectoryCr2"] = withIDCr2Sys["trajectoryID"]
aligned_df["trajectoryCv2"] = withIDCv2Sys["trajectoryID"]
aligned_df["trajectorySys"] = IdSysSys

if myConfig["saveHIResultsToXlsx"]:
    aligned_df.to_excel("data/results.xlsx", index=False)


### ORIGINAL RESULTS

##### CR1 SENSOR SELECTION

In [ ]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr1)}")
for idx,idxSS in enumerate(handledValues):
    ssTable.add_row([idx,idxSS,probsMaskCr1[idx].numpy(),binaryMaskCr1[idx].numpy()])

print(ssTable)

##### CR2 SENSOR SELECTION

In [ ]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr2)}")
for idx,idxSS in enumerate(preData.crusher2Value):
    ssTable.add_row([idx,idxSS,probsMaskCr2[idx].numpy(),binaryMaskCr2[idx].numpy()])

print(ssTable)

##### CV2 SENSOR SELECTION

In [ ]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCv2)}")
for idx,idxSS in enumerate(preData.conveyorValue + preData.crusher2ConveyorValue):
    ssTable.add_row([idx,idxSS,probsMaskCv2[idx].numpy(),binaryMaskCv2[idx].numpy()])

print(ssTable)

#### HI AND RUL VISUALIZATION

In [ ]:
_,mofCr1 = preData.sumarryDataFromFile(cr1)
_,mofCr2 = preData.sumarryDataFromFile(cr2)
_,mofCv2 = preData.sumarryDataFromFile(cv2)

# print((mofCr1))
mof = [mofCr1,mofCr2,mofCv2]
fig = {}
for idxKey,resultKeys in enumerate(results.keys()):
    fig[resultKeys]     = go.Figure()
    current_time        = 0
    # print(math.floor(idxKey/2) + 1)
    for idx, signal in enumerate(results[resultKeys]):
        # print(signal)
        signal          = tf.squeeze(signal).numpy()
        # print(idx)
        # print(signal)
        time_vector     = np.arange(current_time, current_time + len(signal))
        current_time    += len(signal)
        
        # Add each signal as a separate trace
        fig[resultKeys].add_trace(go.Scatter(
                                                x    = time_vector,
                                                y    = signal,
                                                mode = 'lines',
                                                name = f'{idx}. ' + mof[math.floor(idxKey/2)][idx]
        ))
    # Customize layout for readability
    fig[resultKeys].update_layout(
                                    title       = resultKeys,
                                    xaxis_title = "Time (arbitrary units)",
                                    yaxis_title = "Signal Value",
                                    template    = "plotly_white",
                                    showlegend  = True  # Show legend to allow toggling each signal
    )
    # Display the plot
    fig[resultKeys].show()


### FILTERED RESULTS

##### Filter the signal by using moving average

In [18]:
validColumn = ["OriginalHICr1", "OriginalHICr2", "OriginalHICv2"]
trajectory  = ["trajectoryCr1", "trajectoryCr2", "trajectoryCv2", "trajectorySys"]
for idx,col in enumerate(validColumn):
    traject     = trajectory[idx]
    max_traject = aligned_df[traject].max()

    for idxTraject in range(max_traject):
        idxTraject                  += 1
        mask                        = aligned_df[traject] == idxTraject
        aligned_df.loc[mask, col]   = (aligned_df.loc[mask, col].rolling(window=200, min_periods=1, closed='left').mean())

#### Create system signal

In [19]:
scaler = MinMaxScaler()
aligned_df[validColumn] = scaler.fit_transform(aligned_df[validColumn])
aligned_df["System_HI"] = aligned_df[['OriginalHICr1', 'OriginalHICr2', 'OriginalHICv2']].min(axis=1, skipna=False)

In [ ]:
# Threshold for acceptable change
threshold = 0.2
df = aligned_df

# Initialize columns for System_HI and dependency state
df['System_HI'] = np.nan
dependency_state = None  # Initially, no dependency
last_valid_hi = None  # Track the last valid System_HI value

# Iterate over each row
for i in range(len(df)):
    current_row = df.iloc[i]
    components = {
         'HI_Component_1': current_row['OriginalHICr1'],
        'HI_Component_2': current_row['OriginalHICr2'],
        'HI_Component_3': current_row['OriginalHICv2'],
    }

    # Find the current minimum component ignoring NaN
    valid_components = {k: v for k, v in components.items() if not pd.isna(v)}
    if not valid_components:
        # If all components are NaN, assign NaN to System_HI
        df.at[i, 'System_HI'] = np.nan
        continue

    # Find the component with the minimum value
    min_component = min(valid_components, key=valid_components.get)
    min_value = valid_components[min_component]

    if dependency_state is None:
        # First row or initial state: set the dependency state and HI
        dependency_state = min_component
        df.at[i, 'System_HI'] = min_value
        last_valid_hi = min_value
        continue

    # Check if the current dependent component is NaN
    if pd.isna(components[dependency_state]):
        # The current dependent component is NaN, decide on a new state
        if min_value <= last_valid_hi or min_value <= last_valid_hi + threshold:
            # Accept the new state and update the dependency
            dependency_state = min_component
            df.at[i, 'System_HI'] = min_value
            last_valid_hi = min_value
        else:
            # Leave the System_HI as NaN (high peak)
            df.at[i, 'System_HI'] = np.nan
    else:
        # Continue with the current dependency state
        dependency_state = min_component
        df.at[i, 'System_HI'] = min_value
        last_valid_hi = min_value

# Final DataFrame with System_HI
print(df)

#### Visualize

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
# time_filtered_df        = dfFiltered[(dfFiltered['Time'] >= 1961) & (dfFiltered['Time'] <= 3036)]
time_filtered_df        = aligned_df

plotColumn              = ["OriginalHICr1", "OriginalHICr2", "OriginalHICv2","System_HI"]
columns_to_visualize    = ['OriginalHICr1', 'OriginalHICr2', 'OriginalHICv2', 'System_HI']
trajectory_columns      = ['trajectoryCr1', 'trajectoryCr2', 'trajectoryCv2', 'trajectorySys']

# Identify trajectory change points for each trajectory column
transition_points = {}
for trajectory in trajectory_columns:
    transition_indices = np.where(time_filtered_df[trajectory].diff().fillna(0) != 0)[0]
    transition_points[trajectory] = time_filtered_df.iloc[transition_indices]['Time']

# Plot each column with its corresponding trajectory changes
for i, (column, trajectory) in enumerate(zip(columns_to_visualize, trajectory_columns)):
    # Plot the filtered data with gaps for NaN
    # axes[i].plot(time_filtered_df['Time'], time_filtered_df[column], label=column, linestyle='-', marker='o', markersize=2)
    axes[i].plot(time_filtered_df['Time'], time_filtered_df[column], label=column, linewidth=1.5)
    axes[i].set_ylabel('Values')
    # axes[i].legend()
    axes[i].grid(True)
    axes[i].set_xlim([time_filtered_df['Time'].min(), time_filtered_df['Time'].max()])  # Adjust x-axis to the zoomed range

    # Add vertical red lines for trajectory changes corresponding to the trajectory column
    for transition_time in transition_points[trajectory]:
        axes[i].axvline(x=transition_time, color='red', linestyle='--', alpha=0.7)

# Set the x-axis label for the last subplot
axes[-1].set_xlabel('Time')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
time_filtered_df        = aligned_df[(aligned_df['Time'] >= 1961) & (aligned_df['Time'] <= 3036)]
plotColumn              = ["OriginalHICr1", "OriginalHICr2", "OriginalHICv2","System_HI"]
columns_to_visualize    = ['OriginalHICr1', 'OriginalHICr2', 'OriginalHICv2', 'System_HI']
trajectory_columns      = ['trajectoryCr1', 'trajectoryCr2', 'trajectoryCv2', 'trajectorySys']

# Identify trajectory change points for each trajectory column
transition_points = {}
for trajectory in trajectory_columns:
    transition_indices = np.where(time_filtered_df[trajectory].diff().fillna(0) != 0)[0]
    transition_points[trajectory] = time_filtered_df.iloc[transition_indices]['Time']

# Plot each column with its corresponding trajectory changes
for i, (column, trajectory) in enumerate(zip(columns_to_visualize, trajectory_columns)):
    # Plot the filtered data with gaps for NaN
    # axes[i].plot(time_filtered_df['Time'], time_filtered_df[column], label=column, linestyle='-', marker='o', markersize=2)
    axes[i].plot(time_filtered_df['Time'], time_filtered_df[column], label=column, linewidth=1.5)
    axes[i].set_ylabel('Values')
    # axes[i].legend()
    axes[i].grid(True)
    axes[i].set_xlim([time_filtered_df['Time'].min(), time_filtered_df['Time'].max()])  # Adjust x-axis to the zoomed range
    axes[i].set_ylim(0, 1)  # Set y-axis range from 0 to 1 explicitly

    # Add vertical red lines for trajectory changes corresponding to the trajectory column
    for transition_time in transition_points[trajectory]:
        axes[i].axvline(x=transition_time, color='red', linestyle='--', alpha=0.7)

# Set the x-axis label for the last subplot
axes[-1].set_xlabel('Time')

plt.tight_layout()
plt.show()

### EVALUATION

In [ ]:
def proportion_based_monotonicity(hi_series, num_segments=3):
    """
    Calculate proportion-based monotonicity using segments.

    Parameters:
        hi_series (pd.Series): Health indicator series (e.g., HI values over time).
        num_segments (int): Number of segments to divide the data into.

    Returns:
        float: Proportion-based monotonicity (range 0 to 1).
    """
    hi_series = hi_series.dropna()  # Remove NaN values
    
    if len(hi_series) < num_segments:
        return 0.0  # Not enough data for the specified number of segments

    # Step 1: Divide the data into segments and calculate the mean of each segment
    segment_size = len(hi_series) // num_segments
    aggregated = [
        hi_series.iloc[i * segment_size:(i + 1) * segment_size].mean()
        for i in range(num_segments)
    ]
    
    # Step 2: Calculate the proportion of consistent steps
    diffs = pd.Series(aggregated).diff().dropna()  # Calculate differences between segments
    positive_steps = (diffs > 0).sum()
    negative_steps = (diffs < 0).sum()

    # Step 3: Calculate monotonicity as the proportion of consistent steps
    total_steps = len(diffs)
    monotonicity = abs((positive_steps - negative_steps) / total_steps)  # Normalize to 0-1
    # if monotonicity<0.2:
    #     print(monotonicity)
    # if monotonicity == 0:
    #     monotonicity = 1
        # print("testttttttttttttt")
        # print(hi_series)
        # return 1
    # if monotonicity==0:
    #     print("aaaaaaaaaaaaaaaaaaaaaa")
    return monotonicity
def calculate_robustness_polyfit(hi_series, degree=14):
    """
    Calculate robustness using polynomial fitting for trend estimation.

    Parameters:
        hi_series (pd.Series): Health indicator series (e.g., HI values over time).
        degree (int): Degree of the polynomial for trend fitting.

    Returns:
        float: Robustness score (range 0 to 1).
    """
    hi_series = hi_series.dropna()  # Remove NaN values
    n = len(hi_series)
    if n < degree + 1:
        return 0.0  # Not enough data to fit the polynomial

    # Step 1: Fit a polynomial trend
    x = np.arange(n)
    y = hi_series.values
    poly_coeffs = np.polyfit(x, y, degree)  # Fit polynomial
    poly_trend = np.polyval(poly_coeffs, x)  # Evaluate polynomial trend

    # Step 2: Calculate deviations and normalize by standard deviation
    deviations = abs(hi_series - poly_trend)
    std_dev = hi_series.std()
    if std_dev == 0:
        return 1.0  # Perfect stability, no variance

    normalized_deviations = deviations / std_dev

    # Step 3: Compute robustness
    robustness = np.exp(-normalized_deviations.mean())
    return robustness

def calculate_trendability(hi_series, time_step=1):
    """
    Calculate trendability using covariance between time and HI.

    Parameters:
        hi_series (pd.Series): Health indicator series (e.g., HI values over time).
        time_step (float): Step size for generating time values.

    Returns:
        float: Trendability score.
    """
    hi_series = hi_series.dropna()  # Remove NaN values
    n = len(hi_series)
    if n < 2:
        return 0.0  # Not enough data to compute trendability

    # Step 1: Generate time values
    time_values = np.arange(0, n * time_step, time_step)

    # Step 2: Calculate covariance
    covariance = np.cov(time_values, hi_series)[0, 1]

    # Step 3: Return the absolute value of covariance
    trendability = abs(covariance)
    return trendability

# def calculate_normalized_trendability(hi_series, time_step=0.5):
#     """
#     Calculate normalized trendability using covariance between time and HI.

#     Parameters:
#         hi_series (pd.Series): Health indicator series (e.g., HI values over time).
#         time_step (float): Step size for generating time values.

#     Returns:
#         float: Normalized trendability score (range 0 to 1).
#     """
#     hi_series = hi_series.dropna()  # Remove NaN values
#     n = len(hi_series)
#     if n < 2:
#         return 0.0  # Not enough data to compute trendability

#     # Step 1: Generate time values
#     time_values = np.arange(0, n * time_step, time_step)

#     # Step 2: Calculate covariance
#     covariance = np.cov(time_values, hi_series)[0, 1]

#     # Step 3: Normalize by variances
#     var_t = np.var(time_values)
#     var_h = np.var(hi_series)
#     if var_t == 0 or var_h == 0:
#         return 0.0  # Avoid division by zero

#     normalized_trendability = abs(covariance) / (var_t * var_h)
#     return min(1.0, normalized_trendability)  # Clip to range [0, 1]

def calculate_mean_metrics_from_dataframe(df, ID, HI):
    """
    Calculate mean monotonicity and robustness for trajectories in a DataFrame.

    Parameters:
        df (pd.DataFrame): A DataFrame with columns 'ID' and 'HI', where
                           'ID' identifies the trajectory and 'HI' contains the health indicator values.

    Returns:
        dict: Mean monotonicity and robustness across all trajectories.
    """
    grouped = df.groupby(ID)
    
    # Calculate monotonicity and robustness for each trajectory
    # metrics = grouped[HI].apply(lambda hi: pd.Series({
    #     'monotonicity': calculate_monotonicity(hi.values),
    #     'robustness': calculate_robustness(hi.values)
    # }))
    
    metrics = grouped.apply(lambda group: {
        'monotonicity'  : proportion_based_monotonicity(group[HI]),
        'robustness'    : calculate_robustness_polyfit(group[HI]),
        'trendability'  : calculate_trendability(group[HI])
    }).apply(pd.Series)

    # def exclude_zero_min(series):
    #     filtered = series[series != 0]  # Exclude zero values
    #     return filtered.min() if not filtered.empty else None  # Return None if all values are zero

    # Calculate mean and range for each metric
    result_metrics = {
        "monotonicity_mean":    metrics['monotonicity'].mean(),
        "monotonicity_min":     metrics['monotonicity'].min(),
        "monotonicity_max":     metrics['monotonicity'].max(),
        "robustness_mean":      metrics['robustness'].mean(),
        "robustness_min":       metrics['robustness'].min(),
        "robustness_max":       metrics['robustness'].max(),
        "trendability_mean":    metrics['trendability'].mean(),
        "trendability_min":     metrics['trendability'].min(),
        "trendability_max":     metrics['trendability'].max(),
    }

    return result_metrics


trajectory  = ["trajectoryCr1", "trajectoryCr2", "trajectoryCv2", "trajectorySys"]
time_filtered_df        = dfFiltered[(dfFiltered['Time'] >= 0) & (dfFiltered['Time'])]
for idxTraject,idxHI in zip(trajectory,plotColumn):
    # print((df[idxHI]))
    results = calculate_mean_metrics_from_dataframe(dfFiltered,idxTraject,idxHI)
    print(f"Mean Monotonicity {idxHI}: {results['monotonicity_mean']:.3f}")
    print(f"Min Monotonicity {idxHI}: {results['monotonicity_min']:.3f}")
    print(f"Max Monotonicity {idxHI}: {results['monotonicity_max']:.3f}")
    
    print(f"Mean Robustness {idxHI}: {results['robustness_mean']:.3f}")
    print(f"Min Robustness {idxHI}: {results['robustness_min']:.3f}")
    print(f"Max Robustness {idxHI}: {results['robustness_max']:.3f}")

    print(f"Mean Trendability: {results['trendability_mean']:.3f}")
    print(f"Min Trendability: {results['trendability_min']:.3f}")
    print(f"Max Trendability: {results['trendability_max']:.3f}")

In [ ]:
def calculate_effective_monotonicity(hi_series: pd.Series, num_segments: int = 2) -> float:
    """
    Calculate monotonicity using aggregated segments for long-term trends.

    Parameters:
        hi_series (pd.Series): Health indicator series (e.g., HI values over time).
        num_segments (int): Number of segments to divide the data into.

    Returns:
        float: Effective monotonicity score.
    """
    # Step 1: Clean the data
    hi_series = hi_series.dropna()  # Remove NaN values
    
    # Step 2: Handle edge cases
    if len(hi_series) < num_segments:
        return 0.0

    # Step 3: Divide the data into segments and calculate mean for each segment
    segment_size = len(hi_series) // num_segments
    aggregated = pd.Series([
        hi_series.iloc[i * segment_size:(i + 1) * segment_size].mean()
        for i in range(num_segments)
    ])
    
    # Step 4: Calculate slope-based monotonicity on the aggregated values
    def calculate_monotonicity_slope(aggregated_values: pd.Series) -> float:
        diffs = aggregated_values.diff().dropna()
        total_diff = diffs.sum()
        max_possible_diff = diffs.abs().sum()
        return total_diff / max_possible_diff if max_possible_diff != 0 else 0.0

    return calculate_monotonicity_slope(aggregated)

def calculate_lei_monotonicity(hi_series: pd.Series) -> float:
    """
    Calculate monotonicity based on Lei et al. (2018), using positive and negative differences.

    Parameters:
        hi_series (pd.Series): Health indicator sequence.

    Returns:
        float: Monotonicity score (0 to 1).
    """
    hi_series = hi_series.dropna().values  # Remove NaN values
    if len(hi_series) < 2:
        return 0.0  # Not enough data to calculate differences

    # Calculate differences between consecutive points
    dh = np.diff(hi_series)

    # Count positive and negative differences
    num_positive = np.sum(dh > 0)
    num_negative = np.sum(dh < 0)

    # Calculate monotonicity as per Lei et al. (2018)
    total_differences = len(dh)
    monotonicity = abs(num_positive - num_negative) / total_differences

    return monotonicity


def calculate_robustness(hi_series):
    """
    Calculate robustness while skipping NaN values.
    """
    hi_series = hi_series[~np.isnan(hi_series)]  # Remove NaN values
    if len(hi_series) < 2:
        return 0.0
    variance = np.var(hi_series)
    robustness = 1 / (1 + variance)
    return robustness

def calculate_mean_metrics_from_dataframe(df, ID, HI):
    """
    Calculate mean monotonicity and robustness for trajectories in a DataFrame.

    Parameters:
        df (pd.DataFrame): A DataFrame with columns 'ID' and 'HI', where
                           'ID' identifies the trajectory and 'HI' contains the health indicator values.

    Returns:
        dict: Mean monotonicity and robustness across all trajectories.
    """
    grouped = df.groupby(ID)
    
    # Calculate monotonicity and robustness for each trajectory
    # metrics = grouped[HI].apply(lambda hi: pd.Series({
    #     'monotonicity': calculate_monotonicity(hi.values),
    #     'robustness': calculate_robustness(hi.values)
    # }))
    
    metrics = grouped.apply(lambda group: {
        'monotonicity': calculate_lei_monotonicity(group[HI]),
        'robustness': calculate_robustness(group[HI].values)
    }).apply(pd.Series)

    # print(metrics)
    # Calculate mean metrics
    mean_metrics = {
        "monotonicity_mean": metrics['monotonicity'].mean(),
        "robustness_mean": metrics['robustness'].mean()
    }

    return mean_metrics

# test = dfFiltered[dfFiltered["trajectoryCr1"] == 8]["OriginalHICr1"]
# print(test)
# test.to_excel("data/testHI.xlsx", index=False)
# results = calculate_monotonicity(dfFiltered[dfFiltered["trajectoryCr1"] == 8]["OriginalHICr1"])
# print(results)


trajectory  = ["trajectoryCr1", "trajectoryCr2", "trajectoryCv2", "trajectorySys"]
time_filtered_df        = dfFiltered[(dfFiltered['Time'] >= 0) & (dfFiltered['Time'])]
for idxTraject,idxHI in zip(trajectory,plotColumn):
    # print((df[idxHI]))
    results = calculate_mean_metrics_from_dataframe(dfFiltered,idxTraject,idxHI)
    print(f"Mean Monotonicity: {results['monotonicity_mean']:.3f}")
    print(f"Mean Robustness: {results['robustness_mean']:.3f}")